In [2]:
%pip install jaxtyping

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.4 MB/s eta 0:00:00


In [13]:
import os
import sys
from functools import partial
from pathlib import Path
from typing import Callable

import einops
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from ipywidgets import interact
from jaxtyping import Bool, Float
from torch import Tensor
from tqdm.auto import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part1_ray_tracing"

# Clone the repo if we are in Colab and it doesn't exist
if not Path(chapter).exists():
    !git clone https://github.com/callummcdougall/ARENA_3.0.git
    # Move or symlink files if necessary, or update root_dir to the cloned repo
    if Path("ARENA_3.0").exists():
        sys.path.append(str(Path.cwd() / "ARENA_3.0" / chapter / "exercises"))
        root_dir = Path.cwd() / "ARENA_3.0"
else:
    try:
        root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
    except StopIteration:
        root_dir = Path.cwd()
    exercises_dir = root_dir / chapter / "exercises"
    if str(exercises_dir) not in sys.path:
        sys.path.append(str(exercises_dir))

try:
    import part1_ray_tracing.tests as tests
    from part1_ray_tracing.utils import (
        render_lines_with_plotly,
        setup_widget_fig_ray,
        setup_widget_fig_triangle,
    )
    from plotly_utils import imshow
except ModuleNotFoundError:
    # Fallback to importing directly from the repo's path structure if needed
    sys.path.append(str(Path.cwd() / "ARENA_3.0" / chapter / "exercises"))
    import part1_ray_tracing.tests as tests
    from part1_ray_tracing.utils import (
        render_lines_with_plotly,
        setup_widget_fig_ray,
        setup_widget_fig_triangle,
    )
    from plotly_utils import imshow

MAIN = __name__ == "__main__"

In [19]:
from google.colab import output
output.enable_custom_widget_manager()

In [14]:
import os
from pathlib import Path

# Change directory to the exercises subfolder so that Path.cwd().parents contains the root repository directory
correct_dir = "/content/ARENA_3.0/chapter0_fundamentals/exercises"
if Path(correct_dir).exists():
    os.chdir(correct_dir)

def make_rays_1d(num_pixels: int, y_limit: float) -> Tensor:
    """
    num_pixels: The number of pixels in the y dimension. Since there is one ray per pixel, this is
        also the number of rays.
    y_limit: At x=1, the rays should extend from -y_limit to +y_limit, inclusive of both endpoints.

    Returns: shape (num_pixels, num_points=2, num_dim=3) where the num_points dimension contains
        (origin, direction) and the num_dim dimension contains xyz.

    Example of make_rays_1d(9, 1.0): [
        [[0, 0, 0], [1, -1.0, 0]],
        [[0, 0, 0], [1, -0.75, 0]],
        [[0, 0, 0], [1, -0.5, 0]],
        ...
        [[0, 0, 0], [1, 0.75, 0]],
        [[0, 0, 0], [1, 1, 0]],
    ]
    """
    # Create an empty tensor of shape (num_pixels, 2, 3)
    rays = t.zeros((num_pixels, 2, 3))
    # Set the x coordinate of direction (point 1) to 1
    rays[:, 1, 0] = 1.0
    # Generate linear spacing for the y-coordinates from -y_limit to +y_limit
    y_coordinates = t.linspace(-y_limit, y_limit, num_pixels)
    # Assign the y coordinates to the direction (point 1, dimension 1)
    rays[:, 1, 1] = y_coordinates
    return rays

# Run tests and visualize
tests.test_make_rays_1d(make_rays_1d)

rays1d = make_rays_1d(9, 10.0)
fig = render_lines_with_plotly(rays1d)

All tests in `test_make_rays_1d` passed!


In [32]:
def intersect_ray_1d(ray: Float[Tensor, "points dims"], segment: Float[Tensor, "points dims"]) -> bool:
    """
    ray: shape (n_points=2, n_dim=3)  # O, D points
    segment: shape (n_points=2, n_dim=3)  # L_1, L_2 points

    Return True if the ray intersects the segment.
    """

    # Toss the last component of n_dim rq
    ray = ray[:, :-1]
    segment = segment[:, :-1]

    A = t.stack([ray[1], segment[0] - segment[1]], dim = 1)
    b = segment[0] - ray[0]

    try:
      x = t.linalg.solve(A, b)
    except RuntimeError:
      # Bruh i forgot the matrices could be singular
      return False

    return (x[0].item() >= 0) and (x[1].item() >= 0) and (x[1].item() <= 1)

tests.test_intersect_ray_1d(intersect_ray_1d)
tests.test_intersect_ray_1d_special_case(intersect_ray_1d)


All tests in `test_intersect_ray_1d` passed!
All tests in `test_intersect_ray_1d_special_case` passed!
